# 03 - KSC weather-tower analysis for the Falcon 9 explosion

This notebook:

- reads the KSC weather-tower spreadsheet;
- restricts the analysis to the selected explosion interval;
- converts tower heights from feet to metres and wind speeds from knots;
- averages wind vectors correctly using east/north components;
- calculates height-dependent and time-dependent atmospheric profiles;
- estimates a vertically averaged wind over the first 65 m of the acoustic
  path;
- calculates still-air and wind-corrected acoustic speeds and travel times;
- writes summary tables and publication-ready figures.

Wind direction follows the meteorological convention: the direction
**from which** the wind blows.

In [ ]:
# Standard project configuration and isolated output namespace
from pathlib import Path
import sys

NOTEBOOK_NAME = "03_analyze_ksc_weather.ipynb"
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT_HINT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT_HINT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import notebook_context

CTX = notebook_context(NOTEBOOK_NAME, start=CURRENT_DIR)
CONFIG = CTX.config
PROJECT_ROOT = CTX.project_root

# Every notebook writes only inside its own numerically coded namespace.
OUTPUT_DIR = CTX.output_dir
OUTPUT_DATA_DIR = CTX.data_dir
OUTPUT_FIGURE_DIR = CTX.figure_dir
OUTPUT_LOG_DIR = CTX.log_dir

# Backward-compatible aliases used by older cells in this notebook.
DERIVED_OUTPUT_DIR = OUTPUT_DATA_DIR
FIGURE_DIR = OUTPUT_FIGURE_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output: {OUTPUT_DIR}")


### Workflow contract

- Configuration is loaded from `config/project.yml`.
- This notebook writes only to `03_analyze_ksc_weather/` under the configured output root.
- Output filenames carry the `03_` prefix where they are declared explicitly.
- Upstream products are read through the product registry in the YAML file where practical.
- Existing outputs are protected from accidental overwrite by default.


Useful and self-contained.
It produces:
* path-averaged acoustic-speed estimates;
* along-ray wind corrections;
* height and time profiles;
* concise paper values;
* supplementary plots and CSV outputs.
This is a good supporting notebook. It does not need to be embedded tightly in the event-catalogue chain.
I would keep the name and number.
Verdict: keep.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter
from pyproj import Geod

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

MODULE_DIR = PROJECT_ROOT / "modules"
if not MODULE_DIR.exists():
    # This makes the downloaded notebook work alongside the supplied module.
    MODULE_DIR = CURRENT_DIR.parent / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from weather_analysis_fixed import (
    KNOT_TO_MPS,
    along_ray_wind_component,
    calculate_acoustic_summary,
    load_ksc_weather_excel,
    make_height_profile,
    make_time_profile,
    path_average_wind_profile,
    subset_time_window,
    summarize_wind,
)
from physics import compute_speed_of_sound

OUTPUT_DIR = CTX.output_dir
FIGURE_DIR = CTX.figure_dir
DERIVED_DIR = CTX.data_dir
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

## 1. Configuration

In [ ]:
WEATHER_FILE = CONFIG.path("weather_file")

ANALYSIS_START = pd.Timestamp("2016-09-01T13:05:00Z")
ANALYSIS_END = pd.Timestamp("2016-09-01T13:35:00Z")

# Approximate vertical extent of the vehicle.
MAXIMUM_PATH_HEIGHT_M = 65.0

# Final source and receiver coordinates used elsewhere in the project.
SLC40_LAT = 28.56195
SLC40_LON = -80.57719
BCHH_LAT = 28.5740171
BCHH_LON = -80.572375

geod = Geod(ellps="WGS84")
ray_azimuth_deg, _, propagation_distance_m = geod.inv(
    SLC40_LON,
    SLC40_LAT,
    BCHH_LON,
    BCHH_LAT,
)
ray_azimuth_deg %= 360.0

HEIGHT_BIN_M = 10.0

print(f"Weather file: {WEATHER_FILE}")
print(f"Analysis interval: {ANALYSIS_START} to {ANALYSIS_END}")
print(f"SLC-40 → BCHH azimuth: {ray_azimuth_deg:.2f}°")
print(f"SLC-40 → BCHH distance: {propagation_distance_m:.1f} m")

## 2. Load and validate the spreadsheet

In [ ]:
weather_all = load_ksc_weather_excel(WEATHER_FILE)
weather = subset_time_window(
    weather_all,
    ANALYSIS_START,
    ANALYSIS_END,
)

print(f"All rows: {len(weather_all)}")
print(f"Rows in analysis interval: {len(weather)}")
print(
    "Towers/measurement locations:",
    weather["Tower Measurement Location"].nunique(),
)
print(
    "Height range:",
    f"{weather['height_m'].min():.1f}–"
    f"{weather['height_m'].max():.1f} m",
)
print("Observation times:", weather["datetime_utc"].nunique())

display(
    weather[
        [
            "datetime_utc",
            "Tower Measurement Location",
            "height_ft",
            "height_m",
            "wind_direction_from_deg",
            "wind_speed_knots",
            "temperature_f",
            "relative_humidity_percent",
        ]
    ].head()
)

## 3. Overall atmospheric summaries

In [ ]:
wind_summary = summarize_wind(weather)

overall_summary = pd.DataFrame(
    [
        {
            "analysis_start_utc": ANALYSIS_START,
            "analysis_end_utc": ANALYSIS_END,
            "n_rows": len(weather),
            "n_locations": weather[
                "Tower Measurement Location"
            ].nunique(),
            "minimum_height_m": weather["height_m"].min(),
            "maximum_height_m": weather["height_m"].max(),
            "mean_temperature_f": weather["temperature_f"].mean(),
            "std_temperature_f": weather["temperature_f"].std(),
            "n_temperature": weather["temperature_f"].count(),
            "mean_relative_humidity_percent": weather[
                "relative_humidity_percent"
            ].mean(),
            "std_relative_humidity_percent": weather[
                "relative_humidity_percent"
            ].std(),
            "n_humidity": weather[
                "relative_humidity_percent"
            ].count(),
            "scalar_mean_wind_speed_knots": (
                wind_summary.scalar_mean_speed_knots
            ),
            "vector_mean_wind_speed_knots": (
                wind_summary.vector_mean_speed_knots
            ),
            "vector_mean_wind_direction_from_deg": (
                wind_summary.vector_mean_direction_from_deg
            ),
        }
    ]
)

display(overall_summary.T)
overall_summary.to_csv(
    DERIVED_DIR / "weather_overall_summary.csv",
    index=False,
)

## 4. Height and time profiles

In [ ]:
height_profile = make_height_profile(
    weather,
    height_bin_m=HEIGHT_BIN_M,
)
time_profile = make_time_profile(weather)

height_profile["wind_along_slc40_bchh_mps"] = (
    along_ray_wind_component(
        height_profile["mean_u_east_mps"],
        height_profile["mean_v_north_mps"],
        ray_azimuth_deg,
    )
)

display(height_profile)
display(time_profile)

height_profile.to_csv(
    DERIVED_DIR / "weather_height_profile.csv",
    index=False,
)
time_profile.to_csv(
    DERIVED_DIR / "weather_time_profile.csv",
    index=False,
)

## 5. Path-averaged wind and acoustic-speed calculation

In [ ]:
path_wind = path_average_wind_profile(
    height_profile,
    maximum_height_m=MAXIMUM_PATH_HEIGHT_M,
    ray_azimuth_deg=ray_azimuth_deg,
)

acoustic_summary = calculate_acoustic_summary(
    weather,
    height_profile=height_profile,
    maximum_path_height_m=MAXIMUM_PATH_HEIGHT_M,
    ray_azimuth_deg=ray_azimuth_deg,
    propagation_distance_m=propagation_distance_m,
    compute_speed_of_sound=compute_speed_of_sound,
)

acoustic_summary_df = pd.DataFrame([acoustic_summary])
display(acoustic_summary_df.T)

acoustic_summary_df.to_csv(
    DERIVED_DIR / "weather_acoustic_summary.csv",
    index=False,
)

## 6. Wind rose: all average-wind observations

In [ ]:
direction = weather["wind_direction_from_deg"].to_numpy(dtype=float)
speed = weather["wind_speed_knots"].to_numpy(dtype=float)

valid = np.isfinite(direction) & np.isfinite(speed)
direction = direction[valid]
speed = speed[valid]

direction_edges_deg = np.arange(0.0, 360.0 + 22.5, 22.5)
speed_edges_knots = np.array([0, 4, 8, 12, 16, 20, np.inf])

histogram = np.zeros(
    (len(direction_edges_deg) - 1, len(speed_edges_knots) - 1)
)

direction_wrapped = (
    direction + 0.5 * np.diff(direction_edges_deg)[0]
) % 360.0

for i in range(len(direction_edges_deg) - 1):
    direction_mask = (
        (direction_wrapped >= direction_edges_deg[i])
        & (direction_wrapped < direction_edges_deg[i + 1])
    )
    for j in range(len(speed_edges_knots) - 1):
        speed_mask = (
            (speed >= speed_edges_knots[j])
            & (speed < speed_edges_knots[j + 1])
        )
        histogram[i, j] = np.sum(direction_mask & speed_mask)

histogram = 100.0 * histogram / histogram.sum()

theta = np.deg2rad(
    direction_edges_deg[:-1]
    + 0.5 * np.diff(direction_edges_deg)
)
width = np.deg2rad(np.diff(direction_edges_deg))
bottom = np.zeros(len(theta))

fig = plt.figure(figsize=(7.0, 6.5))
ax = fig.add_subplot(111, projection="polar")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)

labels = []
for j in range(histogram.shape[1]):
    values = histogram[:, j]
    ax.bar(
        theta,
        values,
        width=width,
        bottom=bottom,
        align="center",
        edgecolor="white",
        linewidth=0.4,
        label=(
            f"{speed_edges_knots[j]:g}–"
            f"{speed_edges_knots[j + 1]:g} kt"
            if np.isfinite(speed_edges_knots[j + 1])
            else f"≥{speed_edges_knots[j]:g} kt"
        ),
    )
    bottom += values

ax.set_title(
    "KSC average winds, 13:05–13:35 UTC",
    pad=18,
    fontweight="bold",
)
ax.set_ylabel("Frequency (%)")
ax.legend(
    title="Wind speed",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.05),
)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_rose.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_rose.pdf"),
    bbox_inches="tight",
)
plt.show()

## 7. Wind speed versus height

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.0))

ax.scatter(
    weather["wind_speed_knots"],
    weather["height_m"],
    s=8,
    alpha=0.15,
    label="Individual observations",
)
ax.plot(
    height_profile["vector_mean_wind_speed_knots"],
    height_profile["mean_height_m"],
    marker="o",
    linewidth=1.5,
    label="Vector mean by height bin",
)
ax.axhline(
    MAXIMUM_PATH_HEIGHT_M,
    linestyle="--",
    linewidth=1.0,
    label="Approximate rocket height",
)

ax.set_xlabel("Wind speed (knots)")
ax.set_ylabel("Measurement height (m)")
ax.set_title("Wind speed as a function of height")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_speed_vs_height.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_speed_vs_height.pdf"),
    bbox_inches="tight",
)
plt.show()

## 8. Wind direction versus height

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.0))

ax.scatter(
    weather["wind_direction_from_deg"],
    weather["height_m"],
    s=8,
    alpha=0.15,
    label="Individual observations",
)
ax.plot(
    height_profile["vector_mean_wind_direction_from_deg"],
    height_profile["mean_height_m"],
    marker="o",
    linewidth=1.5,
    label="Vector mean by height bin",
)
ax.axhline(
    MAXIMUM_PATH_HEIGHT_M,
    linestyle="--",
    linewidth=1.0,
    label="Approximate rocket height",
)

ax.set_xlim(0, 360)
ax.set_xticks(np.arange(0, 361, 45))
ax.set_xlabel("Meteorological wind direction (° from north)")
ax.set_ylabel("Measurement height (m)")
ax.set_title("Wind direction as a function of height")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_direction_vs_height.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_direction_vs_height.pdf"),
    bbox_inches="tight",
)
plt.show()

## 9. Along-ray wind correction versus height

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.0))

ax.axvline(0.0, linewidth=0.8)
ax.plot(
    height_profile["wind_along_slc40_bchh_mps"],
    height_profile["mean_height_m"],
    marker="o",
    linewidth=1.5,
)
ax.axhline(
    MAXIMUM_PATH_HEIGHT_M,
    linestyle="--",
    linewidth=1.0,
)

ax.set_xlabel(
    "Wind component along SLC-40 → BCHH ray (m s$^{-1}$)"
)
ax.set_ylabel("Measurement height (m)")
ax.set_title("Height-dependent acoustic wind correction")
ax.grid(True, alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_along_ray_wind_vs_height.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_along_ray_wind_vs_height.pdf"),
    bbox_inches="tight",
)
plt.show()

## 10. Temperature versus height

In [ ]:
valid_profile = height_profile.dropna(
    subset=["mean_temperature_c"]
)

fig, ax = plt.subplots(figsize=(6.2, 6.0))
ax.scatter(
    weather["temperature_c"],
    weather["height_m"],
    s=8,
    alpha=0.15,
    label="Individual observations",
)
ax.plot(
    valid_profile["mean_temperature_c"],
    valid_profile["mean_height_m"],
    marker="o",
    linewidth=1.5,
    label="Mean by height bin",
)
ax.axhline(
    MAXIMUM_PATH_HEIGHT_M,
    linestyle="--",
    linewidth=1.0,
    label="Approximate rocket height",
)

ax.set_xlabel("Air temperature (°C)")
ax.set_ylabel("Measurement height (m)")
ax.set_title("Temperature as a function of height")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_temperature_vs_height.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_temperature_vs_height.pdf"),
    bbox_inches="tight",
)
plt.show()

## 11. Relative humidity versus height

In [ ]:
valid_profile = height_profile.dropna(
    subset=["mean_relative_humidity_percent"]
)

fig, ax = plt.subplots(figsize=(6.2, 6.0))
ax.scatter(
    weather["relative_humidity_percent"],
    weather["height_m"],
    s=8,
    alpha=0.15,
    label="Individual observations",
)
ax.plot(
    valid_profile["mean_relative_humidity_percent"],
    valid_profile["mean_height_m"],
    marker="o",
    linewidth=1.5,
    label="Mean by height bin",
)
ax.axhline(
    MAXIMUM_PATH_HEIGHT_M,
    linestyle="--",
    linewidth=1.0,
    label="Approximate rocket height",
)

ax.set_xlabel("Relative humidity (%)")
ax.set_ylabel("Measurement height (m)")
ax.set_title("Relative humidity as a function of height")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_humidity_vs_height.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_humidity_vs_height.pdf"),
    bbox_inches="tight",
)
plt.show()

## 12. Wind speed through time

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))

ax.plot(
    time_profile["datetime_utc"],
    time_profile["vector_mean_wind_speed_knots"],
    marker="o",
    label="Vector mean",
)
ax.plot(
    time_profile["datetime_utc"],
    time_profile["scalar_mean_wind_speed_mps"] / KNOT_TO_MPS,
    marker="s",
    label="Scalar mean",
)

ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Wind speed (knots)")
ax.set_title("Spatially averaged wind speed through time")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_speed_vs_time.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_speed_vs_time.pdf"),
    bbox_inches="tight",
)
plt.show()

## 13. Wind direction through time

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))

ax.plot(
    time_profile["datetime_utc"],
    time_profile["vector_mean_wind_direction_from_deg"],
    marker="o",
)

ax.set_ylim(0, 360)
ax.set_yticks(np.arange(0, 361, 45))
ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Meteorological wind direction (°)")
ax.set_title("Spatially averaged wind direction through time")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_direction_vs_time.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_wind_direction_vs_time.pdf"),
    bbox_inches="tight",
)
plt.show()

## 14. Temperature through time

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))

ax.plot(
    time_profile["datetime_utc"],
    time_profile["mean_temperature_c"],
    marker="o",
)

ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Mean air temperature (°C)")
ax.set_title("Spatially averaged temperature through time")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_temperature_vs_time.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_temperature_vs_time.pdf"),
    bbox_inches="tight",
)
plt.show()

## 15. Relative humidity through time

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))

ax.plot(
    time_profile["datetime_utc"],
    time_profile["mean_relative_humidity_percent"],
    marker="o",
)

ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Mean relative humidity (%)")
ax.set_title("Spatially averaged relative humidity through time")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, alpha=0.25)

fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_humidity_vs_time.png"),
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / CTX.coded_name("weather_humidity_vs_time.pdf"),
    bbox_inches="tight",
)
plt.show()

## 16. Concise values for the paper

In [ ]:
print(
    f"Mean temperature: "
    f"{acoustic_summary['mean_temperature_f']:.2f} °F "
    f"({acoustic_summary['mean_temperature_c']:.2f} °C)"
)
print(
    f"Mean relative humidity: "
    f"{acoustic_summary['mean_relative_humidity_percent']:.1f}%"
)
print(
    f"Still-air sound speed: "
    f"{acoustic_summary['still_air_sound_speed_mps']:.2f} m/s"
)
print(
    f"0–{MAXIMUM_PATH_HEIGHT_M:.0f} m vector-mean wind: "
    f"{acoustic_summary['path_vector_mean_wind_speed_knots']:.2f} kt "
    f"from "
    f"{acoustic_summary['path_vector_mean_wind_direction_from_deg']:.1f}°"
)
print(
    f"Wind component along SLC-40 → BCHH: "
    f"{acoustic_summary['wind_along_ray_mps']:+.2f} m/s"
)
print(
    f"Wind-corrected effective sound speed: "
    f"{acoustic_summary['effective_sound_speed_mps']:.2f} m/s"
)
print(
    f"Still-air travel time: "
    f"{acoustic_summary['still_air_travel_time_s']:.3f} s"
)
print(
    f"Wind-corrected travel time: "
    f"{acoustic_summary['wind_corrected_travel_time_s']:.3f} s"
)
print(
    f"Wind travel-time correction: "
    f"{acoustic_summary['wind_travel_time_correction_s']:+.3f} s"
)